In [3]:
# ========== Preprocessing Data ========== 
# Loading data
import pandas as pd

df = pd.read_csv("vgsales_new.csv")

print(df.shape)
print(df.head())

# Data type conversion
df["Name"] = df["Name"].str.strip()
df["Platform"] = df["Platform"].str.strip()
df["Genre"] = df["Genre"].str.strip()
df["Publisher"] = df["Publisher"].str.strip()

print(df.dtypes)

# Missing values
print(df.isnull().sum())

df["Year"] = df["Year"].fillna(df["Year"].median())
df["Publisher"] = df["Publisher"].fillna("Unknown")

print(df.isnull().sum())

df["Year"] = df["Year"].astype(int)

print(df.dtypes)

# Duplicates
print(df.duplicated().sum())
print(df.duplicated(subset=["Name", "Platform"]).sum())

df.drop_duplicates(subset=["Name", "Platform"], keep="first", inplace=True)

print(df.shape) 

# Outlier detection
Q1 = df["Global_Sales"].quantile(0.25)
Q3 = df["Global_Sales"].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

print(Q1)
print(Q3)
print(IQR)

outliers = df[(df["Global_Sales"] < lower_bound) |
              (df["Global_Sales"] > upper_bound)]

print(len(outliers))

df["Sales_Outlier_Flag"] = (
    (df["Global_Sales"] < lower_bound) | (df["Global_Sales"] > upper_bound)).astype(int)

print(df["Sales_Outlier_Flag"].value_counts())

# Feature engineering
df["Sales_Gap"] = (df["NA_Sales"] - df["EU_Sales"])
df["JP_Share"] = ((df["JP_Sales"] / df["Global_Sales"]) * 100) .round(2)
df["Genre_Encoded"] = df["Genre"].astype("category").cat.codes
df["Platform_Encoded"] = df["Platform"].astype("category").cat.codes

print(df.head())

# Final validation
print(df.shape)
print(df.isnull().sum().sum())
df.to_csv("cleaned_vgsales.csv", index=False)

# ========== EDA ========== 
# Loading cleaned data and imports
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv("cleaned_vgsales.csv")
print(df.shape)

# Sales by region barh chart
plt.figure()
genre_sales = df.groupby("Genre")["Global_Sales"].sum().sort_values()
genre_sales.plot(kind="barh")
plt.title("Global Sales by Genre")
plt.xlabel("Global Sales (millions)")
plt.ylabel("Genre")
plt.tight_layout()

# Global Sales by Genr bar chart
plt.figure()
platform_sales = df.groupby("Platform")["Global_Sales"].sum().sort_values()
platform_sales.plot(kind="barh")
plt.title("Global Sales by Platform")
plt.xlabel("Global Sales (millions)")
plt.ylabel("Platform")
plt.tight_layout()

# Sales Trend Over Time line chart
plt.figure()
global_sales = df.groupby(["Year"])["Global_Sales"].sum().reset_index()
global_sales.plot(kind="line", x="Year", y="Global_Sales")
plt.title("Sales Trend Over Time")
plt.xlabel("Year")
plt.ylabel("Global Sales (millions)")
plt.tight_layout()

FileNotFoundError: [Errno 2] No such file or directory: 'vgsales_new.csv'